In [1]:
import sys
print(f"Python executable: {sys.executable}")

# Check if conda is available
import subprocess
try:
    result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Conda version: {result.stdout.strip()}")
        print("✅ You CAN use !conda commands")
    else:
        print("❌ Conda not found. Use !pip instead")
except FileNotFoundError:
    print("❌ Conda not found. Use !pip instead")

Python executable: C:\Users\ztana\anaconda3\python.exe
✅ Conda version: conda 23.1.0
✅ You CAN use !conda commands


In [ ]:
!conda install -c conda-forge lightgbm -y

In [ ]:
!pip install lightgbm

In [ ]:

# 🌍 PM2.5 Prediction - Complete Modeling Pipeline
## A Beginner-Friendly Guide to Air Quality Prediction

#Welcome! This notebook predicts PM2.5 (fine particulate matter) concentrations using weather and pollution data from cities worldwide. PM2.5 is a key air quality metric that affects human health.

### What You'll Learn:
#- How to explore environmental data
#- How weather impacts air pollution
#- How to compare different machine learning models
#- How to select the best model for prediction

#Let's get started!
#"""


In [ ]:
# =============================================
# PM2.5 Prediction - Complete Modeling Pipeline
# For: Global Weather & Air Pollution Dataset
# =============================================

# %% [markdown]
# ## 1. Install & Import Required Libraries

## 📚 Cell 1: Setup & Import Libraries

# **Purpose:** Load all the tools we need for data analysis and machine learning.

# **Key Concepts:**
# - **Libraries** are pre-built code collections that save us from writing everything from scratch
# - **pandas** → Data manipulation (like Excel for Python)
# - **numpy** → Mathematical operations
# - **matplotlib/seaborn** → Creating charts and graphs
# - **sklearn** → Machine learning models
# - **xgboost/lightgbm** → Advanced prediction models
# - **joblib** → Saving trained models for later use

# **Think of this like:** Gathering all your cooking utensils before starting to cook!
# """

# # %%


# %%
%matplotlib inline

# Run this cell if you're missing any packages
# !pip install pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# For better visualizations - FIXED VERSION
try:
    # Try the new style name (seaborn v0.12+)
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        # Try the old style name (seaborn v0.11 and earlier)
        plt.style.use('seaborn-darkgrid')
    except OSError:
        # Fallback to a default style that always works
        plt.style.use('ggplot')
        print("ℹ️ Using 'ggplot' style as fallback")

# Set seaborn color palette
sns.set_palette("Set2")

print("✅ Libraries imported successfully!")

In [ ]:
# %% [markdown]
# """
# ## 🎯 FEATURE WHITELIST
# **Defines ALL allowed features for modeling.**
# Everything else is automatically excluded!
# """

# %%
# ============================================================
# THE COMPLETE FEATURE WHITELIST
# ============================================================

# %% [markdown]
# """
# ## 🎯 FEATURE WHITELIST - COMPLETE VERSION
# **Explicitly defines which features are ALLOWED and which are FORBIDDEN.**
# """

# %%
print("\n" + "="*60)
print("📋 DEFINING FEATURE WHITELIST")
print("="*60)

# ============================================================
# FORBIDDEN FEATURES (These will be EXCLUDED from the dataset)
# ============================================================

FORBIDDEN_FEATURES = [
    # PM2.5 related (ANY variation)
    'pm2_5', 'pm25', 'target_pm25', 'pm25_category',
    
    # Other pollutants
    'pm10', 'co', 'no', 'no2', 'o3', 'so2', 'nh3',
    
    # AQI and derivatives
    'aqi', 'aqi_label', 'target_aqi_class', 'target_pm10',
    
    # Ratios containing PM2.5
    'pm25_pm10_ratio', 'co_no2_ratio', 'o3_no2_ratio',
    
    # Non-predictive columns
    'weather_time_utc', 'collection_time_utc', 
    'sunrise_utc', 'sunset_utc',
    'city_name', 'country_name', 'country_code', 'continent',
    'geo_lat', 'geo_lon', 'city_population',
    'temp_category', 'wind_compass', 'is_daytime', 
    'comfort_proxy', 'has_rain', 'has_snow'
]

print(f"🚫 Will EXCLUDE {len(FORBIDDEN_FEATURES)} features")

# ============================================================
# ALLOWED FEATURES (These will be KEPT)
# ============================================================

# Weather Features
weather_features = [
    'temp', 'feels_like', 'temp_min', 'temp_max',
    'pressure', 'humidity',
    'wind_speed', 'wind_deg', 'wind_gust',
    'clouds_all', 'visibility',
    'rain_1h', 'rain_3h', 'snow_1h', 'snow_3h'
]

# Time Features
time_features = [
    'year', 'month', 'day', 'hour',
    'day_of_week', 'is_weekend', 'season'
]

# Derived Features (will be created in feature engineering)
derived_features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'wind_deg_sin', 'wind_deg_cos',
    'temp_range', 'hum_temp_interaction', 'wind_speed_sq'
]

ALLOWED_FEATURES = weather_features + time_features + derived_features

print(f"✅ Will KEEP {len(ALLOWED_FEATURES)} features")

print("\n📋 ALLOWED Features:")
print("-"*40)
for f in ALLOWED_FEATURES:
    print(f"  • {f}")

print("\n🚫 FORBIDDEN Features:")
print("-"*40)
for f in FORBIDDEN_FEATURES[:10]:
    print(f"  • {f}")
if len(FORBIDDEN_FEATURES) > 10:
    print(f"  ... and {len(FORBIDDEN_FEATURES) - 10} more")
    

In [ ]:

# %% [markdown]
# ## 2. Load the Dataset

# # %% [markdown]
# """
# ## 📂 Cell 2: Load the Dataset

# **Purpose:** Load the CSV file containing weather and air pollution data.

# **Key Concepts:**
# - **CSV (Comma Separated Values)** is a common file format for tabular data
# - **DataFrame** is pandas' main data structure (think of it as a spreadsheet in Python)
# - **display()** shows a formatted preview of the data

# **Why this matters:** Before analyzing data, we need to bring it into our Python environment. This is like opening a file in Excel before working with it.

# **What we're working with:**
# - 60,715 rows (each row = one observation of a city at a specific time)
# - 88 columns (features like temperature, humidity, PM2.5, etc.)
# - Data from top 3 most populous cities per country worldwide
# """

# %%
# IMPORTANT: Update this path to where your CSV file is located
# Option 1: If the file is in the same folder as your notebook
# **Purpose:** Load data and ONLY keep allowed features + target.


# # %% [markdown]
# """
# ## 📂 Cell 2: Load Dataset (Apply Whitelist)

# **Purpose:** Load data and apply the whitelist.
# **ONLY keeps allowed features, removes forbidden ones.**
# """

# %%
# Load the data
file_path = 'openweather_weather_airpollution_top3cities_per_country.csv'
df = pd.read_csv(file_path)

print(f"📂 Original dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ============================================================
# STEP 1: Find which columns exist in the data
# ============================================================

# Find allowed features that actually exist
available_allowed = [col for col in ALLOWED_FEATURES if col in df.columns]
missing_allowed = [col for col in ALLOWED_FEATURES if col not in df.columns]

print(f"\n✅ Allowed features found: {len(available_allowed)}")
if missing_allowed:
    print(f"⚠️  Allowed features missing ({len(missing_allowed)}):")
    for col in missing_allowed[:5]:
        print(f"  • {col}")
    if len(missing_allowed) > 5:
        print(f"  ... and {len(missing_allowed) - 5} more")

# Find forbidden features that are in the data
forbidden_in_data = [col for col in FORBIDDEN_FEATURES if col in df.columns]
print(f"\n🚫 Forbidden features in data: {len(forbidden_in_data)}")
if forbidden_in_data:
    print("   These will be REMOVED:")
    for col in forbidden_in_data[:10]:
        print(f"  • {col}")
    if len(forbidden_in_data) > 10:
        print(f"  ... and {len(forbidden_in_data) - 10} more")

# ============================================================
# STEP 2: Apply the whitelist
# ============================================================

# Keep ONLY allowed features + target (pm2_5)
final_features = available_allowed + ['pm2_5']
df_clean = df[final_features].copy()

print(f"\n📊 After whitelist: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"   Features: {len(df_clean.columns) - 1}")
print(f"   Target: pm2_5")

# ============================================================
# STEP 3: Verify no forbidden features remain
# ============================================================

print("\n🔍 VERIFICATION:")
forbidden_remaining = [col for col in FORBIDDEN_FEATURES if col in df_clean.columns]
if forbidden_remaining:
    print(f"❌ ERROR: Forbidden features still in data: {forbidden_remaining}")
else:
    print("✅ CONFIRMED: No forbidden features in the data!")

# Show what we're working with
print(f"\n📋 Features being used ({len(df_clean.columns) - 1} features):")
for i, col in enumerate([c for c in df_clean.columns if c != 'pm2_5']):
    print(f"  {i+1:2d}. {col}")

In [ ]:

# %% [markdown]
# ## 3. Initial Data Exploration

# **Purpose:** Understand the basic structure of our data and check for issues.

# **Key Concepts:**
# - **Data Types:** Each column has a type (numbers, text, dates). Different types need different handling.
# - **Missing Values:** Empty cells that need to be filled or handled.
# - **Target Variable:** PM2.5 is what we want to predict (our "target").
# - **Outliers:** Extreme values that might skew our analysis.

# **Why this matters:** You can't build a good model without understanding your data first. This is like checking ingredients before cooking!

# **What we'll discover:**
# - Which columns have missing data
# - How PM2.5 values are distributed
# - The range and typical values of PM2.5
# """

# %%
# Check data types and missing values


print("\n--- Data Info ---")
print(df.info())

# Missing values summary
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_percent
}).sort_values(by='Missing %', ascending=False)
print("\n--- Missing Values (top 20) ---")
print(missing_df.head(20))

# Target variable distribution
print(f"\n--- Target Variable (pm2_5) Summary ---")
print(df['pm2_5'].describe())

# Plot target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['pm2_5'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('PM2.5 Distribution')
axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['pm2_5'].dropna())
axes[1].set_title('PM2.5 Boxplot')
axes[1].set_ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.show()


In [ ]:
# %% [markdown]
# ## 3.5.🔥 NEW: Correlation Heatmap

# **Purpose:** Discover which features are most related to PM2.5.

# **Key Concepts:**
# - **Correlation** measures how two variables move together (range: -1 to +1)
#   - +1 = Perfect positive correlation (as one goes up, the other goes up)
#   - -1 = Perfect negative correlation (as one goes up, the other goes down)
#   - 0 = No relationship
# - **Heatmap** visualizes correlations in a color-coded grid

# **Why this matters:** If we know what's correlated with PM2.5, we can:
# - Understand what causes air pollution (e.g., PM10, NO2 are strongly related)
# - Remove redundant features (e.g., if temp and feels_like are 90% correlated)
# - Focus our modeling on the most important features

# **Think of it like:** Finding out which ingredients are most important for a recipe!
# """

# %%

print("\n" + "="*60)
print("📊 CORRELATION HEATMAP")
print("="*60)

# Select numeric features for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns
# Limit to most relevant features to avoid overcrowding
key_features = ['pm2_5', 'pm10', 'co', 'no2', 'o3', 'so2', 'nh3',
                'temp', 'feels_like', 'humidity', 'pressure', 'wind_speed',
                'clouds_all', 'visibility']

existing_features = [col for col in key_features if col in df.columns]
if len(existing_features) > 1:
    corr_data = df[existing_features].corr()
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5,
                cbar_kws={"shrink": 0.8})
    ax.set_title('Feature Correlation Matrix - PM2.5 Relationships', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Highlight PM2.5 correlations
    pm25_correlations = corr_data['pm2_5'].sort_values(ascending=False)
    print("\n🔍 Features most correlated with PM2.5:")
    print(pm25_correlations.head(10))
else:
    print("⚠️ Not enough numeric features for correlation analysis")
    
    

In [ ]:
# %% [markdown]
# ## 3.6.🔥 NEW: AQI Category Distribution

# **Purpose:** Understand how air quality categories are distributed across our data.

# **Key Concepts:**
# - **AQI (Air Quality Index)** is a simplified scale from 1-5:
#   - 1 = Good (clean air)
#   - 2 = Fair
#   - 3 = Moderate
#   - 4 = Poor
#   - 5 = Very Poor (dangerous air)
# - **Pie Chart** shows proportions/percentages
# - **Boxplot** shows PM2.5 values for each AQI category

# **Why this matters:** AQI gives us a simple way to communicate air quality to non-experts. It also helps us verify that our PM2.5 values make sense (higher AQI should mean higher PM2.5).

# **Think of it like:** Converting temperature to "hot/warm/cool/cold" for easier communication!
# """

# %%


if 'aqi' in df.columns:
    print("\n" + "="*60)
    print("📊 AQI DISTRIBUTION")
    print("="*60)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # AQI distribution (pie chart)
    aqi_counts = df['aqi'].value_counts().sort_index()
    
    # Map AQI values to labels
    aqi_labels = {
        1: 'Good (1)', 2: 'Fair (2)', 3: 'Moderate (3)',
        4: 'Poor (4)', 5: 'Very Poor (5)'
    }
    aqi_labels_used = {k: v for k, v in aqi_labels.items() if k in aqi_counts.index}
    aqi_counts_renamed = aqi_counts.rename(index=aqi_labels_used)
    
    axes[0].pie(aqi_counts_renamed.values, labels=aqi_counts_renamed.index,
               autopct='%1.1f%%', startangle=90, colors=sns.color_palette("RdYlGn_r", len(aqi_counts_renamed)))
    axes[0].set_title('AQI Category Distribution', fontsize=12)
    
    # AQI vs PM2.5 (boxplot)
    sns.boxplot(data=df, x='aqi', y='pm2_5', ax=axes[1])
    axes[1].set_xlabel('AQI Score')
    axes[1].set_ylabel('PM2.5 (µg/m³)')
    axes[1].set_title('PM2.5 Distribution by AQI Category', fontsize=12)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # AQI Statistics
    print("\n📊 AQI Statistics:")
    print(df['aqi'].value_counts().sort_index())
    

In [ ]:

# %% [markdown]
# ## 4. Feature Engineering

# **Purpose:** Create new features from existing ones to help our models make better predictions.

# **Key Concepts:**
# - **Feature Engineering** is creating new variables from existing data
# - **Cyclical Encoding** turns circular data (like hours 0-23) into sine/cosine so models understand that hour 23 and hour 0 are close
# - **Interactions** combine features (e.g., humidity × temperature) to capture combined effects
# - **Ratios** like PM2.5/PM10 show the composition of pollution

# **Why this matters:** The best models are only as good as the features you give them. Feature engineering is where domain knowledge (knowing that PM2.5 varies by hour, season, etc.) translates into better predictions.

# **Think of it like:** A recipe is better if you know not just the ingredients, but how they interact (e.g., baking soda + acid = rise)!

# **What we're creating:**
# - Hour patterns (rush hour vs night)
# - Season patterns (winter vs summer pollution)
# - Weather interactions (humid + hot = different pollution than dry + cold)
# - Pollution composition ratios
# """

# %% [markdown]
"""
## 🔧 Cell 4: Feature Engineering (SAFE VERSION)

**Purpose:** Create new features from ALLOWED features only.
**No PM2.5-related features are created!**
"""

# %%
print("\n" + "="*60)
print("🔧 FEATURE ENGINEERING (Safe Version)")
print("="*60)

def engineer_features_safe(df):
    """
    Create new features from allowed features only.
    NO PM2.5-related features are created!
    """
    df_eng = df.copy()
    
    print("Creating safe (non-leaky) features...")
    
    # 1. Cyclical encoding for time features
    if 'hour' in df_eng.columns:
        df_eng['hour_sin'] = np.sin(2 * np.pi * df_eng['hour'] / 24)
        df_eng['hour_cos'] = np.cos(2 * np.pi * df_eng['hour'] / 24)
        print("  ✅ Created hour_sin/hour_cos")
    
    if 'month' in df_eng.columns:
        df_eng['month_sin'] = np.sin(2 * np.pi * df_eng['month'] / 12)
        df_eng['month_cos'] = np.cos(2 * np.pi * df_eng['month'] / 12)
        print("  ✅ Created month_sin/month_cos")
    
    # 2. Wind direction (cyclical)
    if 'wind_deg' in df_eng.columns:
        df_eng['wind_deg_sin'] = np.sin(2 * np.pi * df_eng['wind_deg'] / 360)
        df_eng['wind_deg_cos'] = np.cos(2 * np.pi * df_eng['wind_deg'] / 360)
        print("  ✅ Created wind_deg_sin/wind_deg_cos")
    
    # 3. Temperature range
    if 'temp_max' in df_eng.columns and 'temp_min' in df_eng.columns:
        df_eng['temp_range'] = df_eng['temp_max'] - df_eng['temp_min']
        print("  ✅ Created temp_range")
    
    # 4. Humidity × Temperature interaction
    if 'humidity' in df_eng.columns and 'temp' in df_eng.columns:
        df_eng['hum_temp_interaction'] = df_eng['humidity'] * df_eng['temp'] / 100
        print("  ✅ Created hum_temp_interaction")
    
    # 5. Wind speed squared
    if 'wind_speed' in df_eng.columns:
        df_eng['wind_speed_sq'] = df_eng['wind_speed'] ** 2
        print("  ✅ Created wind_speed_sq")
    
    # ❌ NOT creating any PM2.5-related features!
    # NO: pm25_pm10_ratio, pm25_category, aqi, etc.
    
    return df_eng

# Apply safe feature engineering
df_eng = engineer_features_safe(df_clean)

print(f"\n📊 After feature engineering: {df_eng.shape[0]:,} rows × {df_eng.shape[1]} columns")

# Show what was created
new_features = [col for col in df_eng.columns if col not in df_clean.columns]
print(f"\n📌 New features created ({len(new_features)}):")
for feature in new_features:
    print(f"  • {feature}")

# Final verification
print("\n🔍 FINAL VERIFICATION:")
leaky_check = ['pm10', 'aqi', 'co', 'no2', 'pm25_category', 'pm25_pm10_ratio']
found = [col for col in leaky_check if col in df_eng.columns]
if found:
    print(f"❌ WARNING: Leaky features still found: {found}")
    print("   Please check your feature engineering!")
else:
    print("✅ CONFIRMED: No leaky features in the data!")
    print("   All features are safe for modeling!")
    

In [ ]:
# %% [markdown]
# ## 4.5.🔥 NEW: Time Series Analysis

# **Purpose:** Understand how PM2.5 changes over time.

# **Key Concepts:**
# - **Time Series** is data collected over time (hourly, daily, etc.)
# - **Moving Average** smooths out short-term fluctuations to show trends
# - **Diurnal Pattern** is the daily cycle (morning rush hour = higher pollution, night = lower)
# - **Trend** is the long-term direction (is pollution getting better or worse?)

# **Why this matters:** Air pollution is strongly time-dependent. Understanding patterns helps us:
# - Identify rush hour effects
# - Detect seasonal trends
# - Validate our data (e.g., are mornings more polluted?)
# - Build better time-based features for prediction

# **Think of it like:** Checking traffic patterns to know when to avoid driving!
# """

# %%

print("\n" + "="*60)
print("📈 TIME SERIES ANALYSIS")
print("="*60)

# Create time series plots
if 'collection_time_utc' in df.columns:
    # Convert to datetime with proper handling of timezone
    df_time = df.copy()
    
    try:
        # First attempt: Convert with timezone handling
        df_time['collection_time_utc'] = pd.to_datetime(df_time['collection_time_utc'], utc=True)
    except (ValueError, TypeError):
        try:
            # Second attempt: Convert without timezone
            df_time['collection_time_utc'] = pd.to_datetime(df_time['collection_time_utc'])
        except (ValueError, TypeError):
            # Third attempt: Try to parse as string
            df_time['collection_time_utc'] = pd.to_datetime(df_time['collection_time_utc'], errors='coerce')
    
    # Drop rows with invalid dates
    df_time = df_time.dropna(subset=['collection_time_utc'])
    
    if len(df_time) > 0:
        # Sort by time for plotting
        df_sorted = df_time.sort_values('collection_time_utc')
        
        fig, axes = plt.subplots(3, 1, figsize=(15, 12))
        
        # Plot 1: PM2.5 over time
        axes[0].plot(df_sorted['collection_time_utc'], df_sorted['pm2_5'], 
                     alpha=0.5, linewidth=0.5, color='steelblue')
        axes[0].set_title('PM2.5 Concentration Over Time', fontsize=14)
        axes[0].set_ylabel('PM2.5 (µg/m³)')
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Moving averages
        if len(df_sorted) > 100:
            # Calculate moving averages with proper handling
            ma_24h = df_sorted['pm2_5'].rolling(window=min(24, len(df_sorted)//4), 
                                               min_periods=1, center=False).mean()
            ma_168h = df_sorted['pm2_5'].rolling(window=min(168, len(df_sorted)//2), 
                                                min_periods=1, center=False).mean()
            
            axes[1].plot(df_sorted['collection_time_utc'], ma_24h, 
                         color='darkred', linewidth=1.5, label='24h Moving Avg')
            axes[1].plot(df_sorted['collection_time_utc'], ma_168h, 
                         color='darkblue', linewidth=2, label='7-day Moving Avg')
            axes[1].set_title('PM2.5 - Moving Averages', fontsize=14)
            axes[1].set_ylabel('PM2.5 (µg/m³)')
            axes[1].grid(True, alpha=0.3)
            axes[1].legend()
        
        # Plot 3: Hourly averages (if hour column exists)
        if 'hour' in df_time.columns and not df_time['hour'].isna().all():
            hourly_avg = df_time.groupby('hour')['pm2_5'].agg(['mean', 'std', 'count']).reset_index()
            
            # Only include hours with sufficient data
            hourly_avg = hourly_avg[hourly_avg['count'] >= 5]
            
            if len(hourly_avg) > 0:
                axes[2].plot(hourly_avg['hour'], hourly_avg['mean'], 'b-', linewidth=2, label='Mean')
                axes[2].fill_between(hourly_avg['hour'], 
                                     hourly_avg['mean'] - hourly_avg['std'],
                                     hourly_avg['mean'] + hourly_avg['std'],
                                     alpha=0.2, color='blue')
                axes[2].set_xlabel('Hour of Day')
                axes[2].set_ylabel('Average PM2.5 (µg/m³)')
                axes[2].set_title('Average PM2.5 by Hour with Standard Deviation', fontsize=14)
                axes[2].grid(True, alpha=0.3)
                axes[2].legend()
            else:
                axes[2].text(0.5, 0.5, 'Insufficient hourly data', 
                           transform=axes[2].transAxes, ha='center')
        
        plt.tight_layout()
        plt.show()
        
        # Additional: Hourly boxplot (if hour column exists)
        if 'hour' in df.columns and not df['hour'].isna().all():
            fig, ax = plt.subplots(figsize=(12, 6))
            df_hourly = df[['hour', 'pm2_5']].dropna()
            
            # Create hourly data for boxplot
            hourly_data = []
            hours_used = []
            for h in range(24):
                hour_data = df_hourly[df_hourly['hour'] == h]['pm2_5'].dropna()
                if len(hour_data) > 5:  # Only include hours with sufficient data
                    hourly_data.append(hour_data)
                    hours_used.append(h)
            
            if len(hourly_data) > 0:
                ax.boxplot(hourly_data, positions=hours_used)
                ax.set_xlabel('Hour of Day')
                ax.set_ylabel('PM2.5 (µg/m³)')
                ax.set_title('PM2.5 Distribution by Hour (Diurnal Pattern)', fontsize=14)
                ax.set_xticks(range(0, 24, 3))
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            else:
                print("⚠️ Insufficient data for hourly boxplot analysis")
        
        # Print time range information
        print(f"\n📅 Data time range:")
        print(f"   Start: {df_sorted['collection_time_utc'].min()}")
        print(f"   End: {df_sorted['collection_time_utc'].max()}")
        print(f"   Total records: {len(df_sorted):,}")
        
    else:
        print("⚠️ No valid dates found after processing")
else:
    print("⚠️ 'collection_time_utc' column not found. Skipping time series analysis.")
    

In [ ]:
# %% [markdown]
# ## 4.6.🔥 NEW: Seasonal Analysis

# **Purpose:** Understand how PM2.5 varies across seasons and months.

# **Key Concepts:**
# - **Seasonality** is patterns that repeat at regular intervals (e.g., every year)
# - **Boxplot by Season** shows how PM2.5 distribution changes in winter vs summer
# - **Monthly Averages** shows more granular seasonal patterns

# **Why this matters:** Air pollution often varies dramatically by season:
# - Winter = more heating, temperature inversions trap pollution
# - Summer = more ozone formation from sunlight
# - Spring/Fall = transitional patterns

# **Think of it like:** Understanding that you need to dress differently in different seasons - pollution also "dresses" differently depending on the time of year!

# **What this reveals:**
# - When is air quality worst?
# - What drives seasonal differences?
# - Can we use this for better predictions?
# """

# %%

print("\n" + "="*60)
print("📊 SEASONAL ANALYSIS")
print("="*60)

if 'season' in df.columns and not df['season'].isna().all():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: PM2.5 by season (boxplot)
    season_order = ['Winter', 'Spring', 'Summer', 'Fall'] 
    # Check which seasons actually exist in the data
    available_seasons = [s for s in season_order if s in df['season'].unique()]
    if len(available_seasons) > 0:
        sns.boxplot(data=df, x='season', y='pm2_5', order=available_seasons, ax=axes[0])
        axes[0].set_title('PM2.5 Distribution by Season', fontsize=12)
        axes[0].set_ylabel('PM2.5 (µg/m³)')
        axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Monthly averages
    if 'month' in df.columns and not df['month'].isna().all():
        monthly_avg = df.groupby('month')['pm2_5'].agg(['mean', 'std']).reset_index()
        axes[1].bar(monthly_avg['month'], monthly_avg['mean'], 
                   color='steelblue', alpha=0.7, yerr=monthly_avg['std'], capsize=5)
        axes[1].set_xlabel('Month')
        axes[1].set_ylabel('Average PM2.5 (µg/m³)')
        axes[1].set_title('Monthly Average PM2.5 with Standard Deviation', fontsize=12)
        axes[1].set_xticks(range(1, 13))
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics by season
    print("\n📊 PM2.5 Statistics by Season:")
    print(df.groupby('season')['pm2_5'].describe())
else:
    print("⚠️ 'season' column not found. Skipping seasonal analysis.")
    

In [ ]:
# %% [markdown]
# ## 4.7.🔥 NEW: Geographic Distribution Analysis

# **Purpose:** Visualize where pollution is highest and lowest across the world.

# **Key Concepts:**
# - **Geographic Visualization** shows data on a map-like scatter plot
# - **Latitude/Longitude** are coordinates that position cities on Earth
# - **Color Mapping** represents PM2.5 values (green = clean, red = polluted)
# - **City Ranking** identifies the most and least polluted cities

# **Why this matters:** Air pollution is not evenly distributed globally. Understanding geography helps us:
# - Identify regions with severe air quality problems
# - Target policy interventions
# - Understand regional pollution sources (industrial, desert dust, etc.)

# **Think of it like:** A weather map, but for air quality!

# **What this reveals:**
# - Which countries/cities have the worst air quality?
# - Are there regional patterns (e.g., Asia vs Europe)?
# - Which cities could serve as "clean air" benchmarks?
# """

# %%

print("\n" + "="*60)
print("🌍 GEOGRAPHIC DISTRIBUTION")
print("="*60)

# Check if we have latitude and longitude
if 'geo_lat' in df.columns and 'geo_lon' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Plot 1: Scatter map
    scatter = axes[0].scatter(df['geo_lon'], df['geo_lat'],
                             c=df['pm2_5'], cmap='RdYlGn_r',
                             s=20, alpha=0.6, edgecolors='black', linewidth=0.3)
    
    cbar = plt.colorbar(scatter, ax=axes[0])
    cbar.set_label('PM2.5 (µg/m³)', fontsize=12)
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].set_title('Geographic Distribution of PM2.5 Concentrations', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Top 10 Most Polluted vs Cleanest Cities
    if 'city_name' in df.columns:
        # Calculate average PM2.5 per city
        city_avg = df.groupby('city_name')['pm2_5'].agg(['mean', 'count']).reset_index()
        city_avg = city_avg[city_avg['count'] >= 5]  # Only cities with at least 5 observations
        
        if len(city_avg) >= 10:
            # Get top 10 most and least polluted
            top_10_polluted = city_avg.nlargest(10, 'mean')
            top_10_clean = city_avg.nsmallest(10, 'mean')
            
            # Combine for plotting
            top_10_polluted['category'] = 'Most Polluted'
            top_10_clean['category'] = 'Cleanest'
            city_combined = pd.concat([top_10_polluted, top_10_clean])
            
            colors = ['darkred' if c == 'Most Polluted' else 'forestgreen' for c in city_combined['category']]
            bars = axes[1].barh(range(len(city_combined)), city_combined['mean'], color=colors, alpha=0.7)
            axes[1].set_yticks(range(len(city_combined)))
            axes[1].set_yticklabels(city_combined['city_name'])
            axes[1].set_xlabel('Average PM2.5 (µg/m³)')
            axes[1].set_title('Top 10 Most Polluted vs Cleanest Cities', fontsize=12)
            axes[1].grid(True, alpha=0.3)
            
            # Add legend
            from matplotlib.patches import Patch
            legend_elements = [Patch(facecolor='darkred', label='Most Polluted'),
                              Patch(facecolor='forestgreen', label='Cleanest')]
            axes[1].legend(handles=legend_elements)
    
    plt.tight_layout()
    plt.show()
    
    # Summary by continent
    if 'continent' in df.columns and not df['continent'].isna().all():
        print("\n📊 Average PM2.5 by Continent:")
        print(df.groupby('continent')['pm2_5'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False))
else:
    print("⚠️ Geographic columns not found. Skipping geographic analysis.")

In [ ]:
# %% [markdown]
# ## 4.8.🔥 NEW: Weather Impact Analysis

# **Purpose:** Understand how different weather conditions affect PM2.5.

# **Key Concepts:**
# - **Scatter Plot** shows relationship between two variables
# - **Regression Line** (R²) shows how much one variable predicts another
# - **p-value** tells us if the relationship is statistically significant

# **Why this matters:** Weather is one of the biggest drivers of air pollution:
# - **Wind** disperses pollution (windy = cleaner air)
# - **Rain** washes pollution out of the air
# - **Temperature** affects chemical reactions (e.g., ozone formation)
# - **Humidity** affects particle formation and visibility

# **Think of it like:** Understanding how weather affects a garden - some conditions (sun, water) help plants grow, others (wind, frost) hurt them. Similarly, some weather conditions "help" pollution grow, others "hurt" it!

# **What this reveals:**
# - Does wind actually clean the air? (Should be negative correlation)
# - Does temperature increase or decrease pollution?
# - How strong are these relationships?

# **R² Interpretation:**
# - 0.00 = no relationship
# - 0.25 = weak relationship
# - 0.50 = moderate relationship
# - 0.75+ = strong relationship
# """

# %%


print("\n" + "="*60)
print("🌤️ WEATHER IMPACT ANALYSIS")
print("="*60)

# Scatter plots showing relationship between weather and PM2.5
weather_features = ['temp', 'humidity', 'wind_speed', 'pressure', 'visibility']
existing_weather = [col for col in weather_features if col in df.columns]

if len(existing_weather) >= 2:
    # Determine grid size
    n_plots = len(existing_weather)
    n_cols = 2
    n_rows = (n_plots + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5*n_rows))
    if n_rows == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    from scipy import stats
    
    for i, feature in enumerate(existing_weather):
        if i < len(axes) and not df[feature].isna().all():
            # Scatter plot
            axes[i].scatter(df[feature], df['pm2_5'], alpha=0.2, s=5)
            
            # Add trend line
            mask = ~(df[feature].isna() | df['pm2_5'].isna())
            if mask.sum() > 10:
                slope, intercept, r_value, p_value, std_err = stats.linregress(
                    df[feature][mask], df['pm2_5'][mask]
                )
                x_range = np.linspace(df[feature][mask].min(), df[feature][mask].max(), 100)
                y_range = slope * x_range + intercept
                axes[i].plot(x_range, y_range, 'r-', linewidth=2)
                axes[i].text(0.05, 0.95, f'r² = {r_value**2:.3f}\np = {p_value:.4f}', 
                           transform=axes[i].transAxes, 
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                           verticalalignment='top')
            
            axes[i].set_xlabel(feature.capitalize())
            axes[i].set_ylabel('PM2.5 (µg/m³)')
            axes[i].set_title(f'PM2.5 vs {feature.capitalize()}', fontsize=12)
            axes[i].grid(True, alpha=0.3)
    
    # Hide empty subplots
    for i in range(len(existing_weather), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation with weather features
    print("\n📊 Correlation between PM2.5 and Weather Features:")
    weather_corr = df[['pm2_5'] + existing_weather].corr()['pm2_5'].sort_values(ascending=False)
    print(weather_corr)
else:
    print("⚠️ Not enough weather features for analysis")

In [ ]:
# %% [markdown]
# %% [markdown]
# ## 4.9.🔥 Clean Feature Preparation: Remove features that directly relate to PM2.5.


# """
# ## 🧹 Cell: Clean Feature Preparation (NO DATA LEAKAGE)
# **Purpose:** Remove features that directly relate to PM2.5.
# """

# %%
print("\n" + "="*60)
print("🔧 PREPARING CLEAN FEATURES FOR REAL PM2.5 PREDICTION")
print("="*60)

def prepare_clean_data(df, target='pm2_5'):
    """
    Prepare data with NO leaky features
    """
    data = df.copy()
    
    # 1. Remove ALL PM2.5-related features
    leaky_features = [
        # Target variants
        'pm2_5', 'target_pm25', 'pm25_category',
        
        # Other pollutants (strongly correlated)
        'pm10', 'co', 'no', 'no2', 'o3', 'so2', 'nh3',
        
        # AQI and derivatives
        'aqi', 'aqi_label', 'target_aqi_class', 'target_pm10',
        
        # Ratios containing PM2.5
        'pm25_pm10_ratio', 'co_no2_ratio', 'o3_no2_ratio',
        
        # Non-predictive columns
        'weather_time_utc', 'collection_time_utc', 'sunrise_utc', 'sunset_utc',
        'city_name', 'country_name', 'country_code', 'continent',
        'geo_lat', 'geo_lon', 'city_population',
        'temp_category', 'wind_compass', 'is_daytime', 'comfort_proxy',
        'has_rain', 'has_snow'
    ]
    
    # Keep only weather and time features
    weather_features = ['temp', 'feels_like', 'temp_min', 'temp_max', 
                        'pressure', 'humidity', 'wind_speed', 'wind_deg', 
                        'wind_gust', 'clouds_all', 'visibility', 
                        'rain_1h', 'rain_3h', 'snow_1h', 'snow_3h']
    
    time_features = ['year', 'month', 'day', 'hour', 'day_of_week', 
                     'is_weekend', 'season']
    
    # Cyclical encoded features
    cyclical_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos',
                         'wind_deg_sin', 'wind_deg_cos']
    
    # Derived features (valid ones)
    valid_features = ['temp_range', 'hum_temp_interaction', 'wind_speed_sq']
    
    # Keep only valid features
    keep_features = weather_features + time_features + cyclical_features + valid_features
    keep_features = [col for col in keep_features if col in data.columns]
    
    # Create clean dataset
    X = data[keep_features].copy()
    y = data[target].copy()
    
    print(f"✅ Clean features ({len(keep_features)} total):")
    for i, feature in enumerate(keep_features):
        print(f"  {i+1:2d}. {feature}")
    
    print(f"\n🎯 Target: {target}")
    
    return X, y, keep_features

# Prepare clean data
X_clean, y_clean, clean_features = prepare_clean_data(df_eng)

# Check for missing values
print("\n📊 Missing values after cleaning:")
print(X_clean.isnull().sum().sort_values(ascending=False).head(10))

# Handle missing values
for col in X_clean.columns:
    if X_clean[col].isnull().any():
        X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"\n✅ Final clean dataset: {X_clean.shape[0]:,} samples with {X_clean.shape[1]} valid features")

In [ ]:
## 🧹 Section 5-8: Data Preparation for Modeling

# **Purpose:** Prepare our data for machine learning models.

# **Key Concepts:**

# **Section 5 (Prepare Model Data):**
# - **Feature/Target Split:** Separate features (X) from what we want to predict (y)
# - **Categorical Encoding:** Convert text categories (like 'season') to numbers
# - **Missing Value Handling:** Fill in missing values with medians (robust to outliers)

# **Section 6 (Train-Test Split):**
# - **Time-Based Split:** Train on older data, test on newer data (realistic)
# - **Random Split:** Randomly assign data to training/testing sets
# - **Why?** We need to test our model on data it hasn't seen before

# **Section 7 (Feature Scaling):**
# - **Standardization:** Transform features to have mean=0 and std=1
# - **Why?** Some models (like Ridge) need features on the same scale

# **Section 8 (Evaluation Function):**
# - **MAE (Mean Absolute Error):** Average prediction error (in µg/m³)
# - **RMSE (Root Mean Squared Error):** Penalizes large errors more
# - **R² Score:** How much variance we explain (0-1 scale)

# **Why this matters:** Good data preparation is like prepping ingredients before cooking - if you don't do it right, the final dish won't taste good!
# """

In [ ]:

# %% [markdown]
# ## 5. Prepare Data for Modeling

# **Key Concepts:**

# - **Feature/Target Split:** Separate features (X) from what we want to predict (y)
# - **Categorical Encoding:** Convert text categories (like 'season') to numbers
# - **Missing Value Handling:** Fill in missing values with medians (robust to outliers)

# %%
def prepare_model_data(df, target='pm2_5'):
    """
    Prepare data for modeling with proper handling of categorical and numeric features
    """
    # Make a copy
    data = df.copy()
    
    # Drop columns that shouldn't be used as features
    # These are identifiers, coordinates, or other non-predictive columns
    drop_columns = [
        'weather_time_utc', 'collection_time_utc', 'sunrise_utc', 'sunset_utc',
        'city_name', 'country_name', 'country_code', 'continent',
        'geo_lat', 'geo_lon', 'city_population',
        # We'll keep these but they might need special handling
        'target_aqi_class', 'target_pm25', 'target_pm10'
    ]
    
    # Only drop columns that exist
    cols_to_drop = [col for col in drop_columns if col in data.columns]
    data = data.drop(columns=cols_to_drop, errors='ignore')
    
    # Separate features and target
    if target not in data.columns:
        raise ValueError(f"Target column '{target}' not found in dataset")
    
    y = data[target].copy()
    X = data.drop(columns=[target], errors='ignore')
    
    # Handle categorical columns
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    print(f"\nCategorical columns: {categorical_cols}")
    
    # Encode categorical columns
    for col in categorical_cols:
        if col in X.columns:
            le = LabelEncoder()
            # Handle NaN values in categorical columns
            X[col] = X[col].fillna('Unknown')
            X[col] = le.fit_transform(X[col].astype(str))
    
    # Handle missing values in numeric columns
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X[col].isnull().any():
            # Fill with median (more robust than mean)
            X[col] = X[col].fillna(X[col].median())
    
    # Drop rows where target is NaN
    mask = ~y.isna()
    X = X[mask]
    y = y[mask]
    
    print(f"\nFinal dataset shape: X={X.shape}, y={y.shape}")
    print(f"Number of features: {X.shape[1]}")
    
    return X, y, categorical_cols

# Prepare the data
X, y, categorical_cols = prepare_model_data(df_eng, target='pm2_5')

# Display feature names
print(f"\nFeatures ({len(X.columns)} total):")
print(X.columns.tolist()[:20])  # Show first 20

In [ ]:

# %% [markdown]
# ## 6. Train-Test Split (Time-Based)

# **Key Concepts:**

# - **Time-Based Split:** Train on older data, test on newer data (realistic)
# - **Random Split:** Randomly assign data to training/testing sets
# - **Why?** We need to test our model on data it hasn't seen before

# %%
# Check if we have a date column to do time-based split
# If not, we'll do a random split

def split_data(X, y, df_original, test_size=0.2):
    """
    Split data either randomly or time-based if date column exists
    """
    # Try to use time-based split if we have collection_time_utc
    if 'collection_time_utc' in df_original.columns:
        print("Using time-based train-test split...")
        # Convert to datetime if needed
        dates = pd.to_datetime(df_original['collection_time_utc'])
        # Get the index from X to align
        X_indices = X.index
        dates_aligned = dates.loc[X_indices]
        
        # Sort by date
        sorted_idx = dates_aligned.sort_values().index
        X_sorted = X.loc[sorted_idx]
        y_sorted = y.loc[sorted_idx]
        
        # Split by time (first 80% for training, last 20% for testing)
        split_idx = int(len(X_sorted) * (1 - test_size))
        
        X_train = X_sorted.iloc[:split_idx]
        X_test = X_sorted.iloc[split_idx:]
        y_train = y_sorted.iloc[:split_idx]
        y_test = y_sorted.iloc[split_idx:]
        
        print(f"Training period: {dates_aligned.loc[X_train.index].min()} to {dates_aligned.loc[X_train.index].max()}")
        print(f"Testing period: {dates_aligned.loc[X_test.index].min()} to {dates_aligned.loc[X_test.index].max()}")
        
        return X_train, X_test, y_train, y_test
    
    else:
        print("No date column found. Using random split...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_data(X, y, df_eng)

print(f"\nTraining set: {len(X_train)} samples")
print(f"Testing set: {len(X_test)} samples")

In [ ]:

# %% [markdown]
# ## 7. Feature Scaling (for models that need it)

# **Key Concepts:**
# - **Standardization:** Transform features to have mean=0 and std=1
# - **Why?** Some models (like Ridge) need features on the same scale

# %%
# Scale features for models like Ridge, LSTM, etc.
# Tree-based models (Random Forest, XGBoost, LightGBM) don't need scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# For tree-based models, keep unscaled version
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

print("Scaling completed. Ready for modeling.")

# %% [markdown]
# ## 8. Model Training and Evaluation

# %%
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    Evaluate a model and return metrics
    """
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n--- {model_name} Performance ---")
    print(f"MAE: {mae:.4f} µg/m³")
    print(f"RMSE: {rmse:.4f} µg/m³")
    print(f"R² Score: {r2:.4f}")
    
    return {
        'model': model,
        'name': model_name,
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'predictions': y_pred
    }


In [ ]:
# **Section 8 (Evaluation Function):**

# **Key Concepts:**

# - **MAE (Mean Absolute Error):** Average prediction error (in µg/m³)
# - **RMSE (Root Mean Squared Error):** Penalizes large errors more
# - **R² Score:** How much variance we explain (0-1 scale)

## 🤖 Training Different Models

# **Purpose:** Compare different machine learning models to find the best performer.

# **Why compare models?** Different algorithms work better for different data types. 
# **We're testing which one is best for our specific air pollution data.
# **Think of it like:** Trying different tools for a job - a hammer might work, but a screwdriver might be better!
    

In [ ]:
# %% [markdown]
# ### 8.1 Ridge Regression (Linear Baseline)

# **Ridge Regression (Linear Model):**
# - **What:** Simple linear equation with regularization
# - **Pros:** Fast, interpretable, good baseline
# - **Cons:** Can only capture linear relationships    

# %%
print("="*60)
print("Training Ridge Regression...")
print("="*60)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_results = evaluate_model(ridge, X_test_scaled, y_test, "Ridge Regression")

# Plot predictions vs actual
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(y_test, ridge_results['predictions'], alpha=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual PM2.5')
ax.set_ylabel('Predicted PM2.5')
ax.set_title(f'Ridge Regression: Predictions vs Actual (R²={ridge_results["r2"]:.4f})')
plt.show()


In [ ]:
# %% [markdown]
# ### 8.2 Random Forest

# **Random Forest (Tree Ensemble):**
# - **What:** Many decision trees working together
# - **Pros:** Handles non-linear relationships, no scaling needed
# - **Cons:** Slower, can overfit
    
# %%
print("="*60)
print("Training Random Forest...")
print("="*60)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_tree, y_train)
rf_results = evaluate_model(rf, X_test_tree, y_test, "Random Forest")

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X_train_tree.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Features (Random Forest) ---")
print(feature_importance.head(15))

# Plot feature importance
plt.figure(figsize=(12, 8))
plt.barh(feature_importance.head(15)['feature'], feature_importance.head(15)['importance'])
plt.xlabel('Importance')
plt.title('Random Forest - Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# %% [markdown]
# ### 8.3 XGBoost

# **XGBoost (Gradient Boosting):**
# - **What:** Sequential tree building, each tree corrects previous errors
# - **Pros:** State-of-the-art performance, handles missing data
# - **Cons:** More parameters to tune
    
# %%
print("="*60)
print("Training XGBoost...")
print("="*60)

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_tree, y_train)
xgb_results = evaluate_model(xgb_model, X_test_tree, y_test, "XGBoost")

# XGBoost Feature Importance
xgb_importance = pd.DataFrame({
    'feature': X_train_tree.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Features (XGBoost) ---")
print(xgb_importance.head(15))



In [ ]:
# %% [markdown]
# ### 8.4 LightGBM

# **LightGBM (Fast Gradient Boosting):**
# - **What:** Similar to XGBoost but faster
# - **Pros:** Very fast, memory efficient, similar performance
# - **Cons:** Can overfit on small datasets
    
# %%
print("="*60)
print("Training LightGBM...")
print("="*60)

lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train_tree, y_train)
lgb_results = evaluate_model(lgb_model, X_test_tree, y_test, "LightGBM")

# LightGBM Feature Importance
lgb_importance = pd.DataFrame({
    'feature': X_train_tree.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Features (LightGBM) ---")
print(lgb_importance.head(15))

In [ ]:

# %% [markdown]
# ## 9. Model Comparison

# **Purpose:** Compare all models and select the best one.

"""
**Key Concepts:**
- **Model Comparison** helps us choose the best algorithm
- **MAE (Mean Absolute Error):** Average error in µg/m³ (easier to interpret)
- **RMSE:** Penalizes large errors (sensitive to outliers)
- **R²:** How much variance we explain (0-1 scale)

**Why this matters:** The "best" model depends on your goal:
- If you need simplicity → Ridge (easy to explain)
- If you need accuracy → XGBoost or LightGBM
- If you need robustness → Random Forest

**Think of it like:** Choosing a vehicle - a bicycle, car, or truck each has different strengths!
"""

# %%
# Collect all results
all_results = [ridge_results, rf_results, xgb_results, lgb_results]

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': [r['name'] for r in all_results],
    'MAE (µg/m³)': [r['mae'] for r in all_results],
    'RMSE (µg/m³)': [r['rmse'] for r in all_results],
    'R² Score': [r['r2'] for r in all_results]
})

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(comparison_df.to_string(index=False))

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['MAE (µg/m³)', 'RMSE (µg/m³)', 'R² Score']
for i, metric in enumerate(metrics):
    axes[i].bar(comparison_df['Model'], comparison_df[metric])
    axes[i].set_title(metric)
    axes[i].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# %% [markdown]
# ## 10. Save Best Model

# %%
# Determine the best model (based on lowest RMSE)
best_idx = comparison_df['RMSE (µg/m³)'].idxmin()
best_model = all_results[best_idx]['model']
best_name = all_results[best_idx]['name']

print(f"\n🏆 Best Model: {best_name}")
print(f"   RMSE: {comparison_df.loc[best_idx, 'RMSE (µg/m³)']:.4f} µg/m³")
print(f"   R²: {comparison_df.loc[best_idx, 'R² Score']:.4f}")

# Save the best model
import joblib
joblib.dump(best_model, 'best_pm25_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print(f"\n✅ Best model saved as 'best_pm25_model.pkl'")
print(f"✅ Scaler saved as 'scaler.pkl'")


In [ ]:

# %% [markdown]
# ## 11. Make Predictions on New Data

# %%
def predict_pm25(model, scaler, new_data):
    """
    Make PM2.5 predictions on new data
    new_data should be a DataFrame with the same features as training data
    """
    # Apply the same feature engineering
    new_data_eng = engineer_features(new_data)
    
    # Align columns with training data
    # Reorder columns to match training
    training_columns = X_train_tree.columns
    for col in training_columns:
        if col not in new_data_eng.columns:
            new_data_eng[col] = 0  # Fill missing columns with 0
    
    # Select only training columns
    new_data_final = new_data_eng[training_columns]
    
    # Scale if using ridge model (for tree models, not necessary but safe)
    new_data_scaled = scaler.transform(new_data_final)
    
    # Make predictions
    predictions = model.predict(new_data_scaled)
    
    return predictions

# Example: Use the test set to demonstrate
print("\n--- Example Predictions on Test Data ---")
sample_indices = X_test_tree.index[:10]
sample_predictions = best_model.predict(X_test_tree.loc[sample_indices])

for i, idx in enumerate(sample_indices):
    actual = y_test.loc[idx]
    pred = sample_predictions[i]
    print(f"Sample {i+1}: Actual={actual:.2f}, Predicted={pred:.2f}, Error={abs(actual-pred):.2f}")

print("\n" + "="*60)
print("✅ Modeling pipeline complete!")
print("="*60)

In [ ]:
# 🔥 NEW: Enhanced Feature Importance Visualization

print("\n" + "="*60)
print("📊 ENHANCED FEATURE IMPORTANCE VISUALIZATION")
print("="*60)

if 'best_model' in globals():
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    # Get feature importances from all tree-based models
    models_importance = []
    
    # Random Forest
    if 'rf' in globals():
        rf_imp = pd.DataFrame({
            'feature': X_train_tree.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False).head(10)
        axes[0].barh(rf_imp['feature'], rf_imp['importance'], color='green')
        axes[0].set_title('Random Forest - Top 10 Features', fontsize=12)
        axes[0].set_xlabel('Importance')
        axes[0].invert_yaxis()
        axes[0].grid(True, alpha=0.3)
    
    # XGBoost
    if 'xgb_model' in globals():
        xgb_imp = pd.DataFrame({
            'feature': X_train_tree.columns,
            'importance': xgb_model.feature_importances_
        }).sort_values('importance', ascending=False).head(10)
        axes[1].barh(xgb_imp['feature'], xgb_imp['importance'], color='blue')
        axes[1].set_title('XGBoost - Top 10 Features', fontsize=12)
        axes[1].set_xlabel('Importance')
        axes[1].invert_yaxis()
        axes[1].grid(True, alpha=0.3)
    
    # LightGBM
    if 'lgb_model' in globals():
        lgb_imp = pd.DataFrame({
            'feature': X_train_tree.columns,
            'importance': lgb_model.feature_importances_
        }).sort_values('importance', ascending=False).head(10)
        axes[2].barh(lgb_imp['feature'], lgb_imp['importance'], color='orange')
        axes[2].set_title('LightGBM - Top 10 Features', fontsize=12)
        axes[2].set_xlabel('Importance')
        axes[2].invert_yaxis()
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Compare feature importance across models
    print("\n📊 Top 5 Features by Each Model:")
    if 'rf_imp' in locals():
        print("\nRandom Forest:", rf_imp['feature'].head(5).tolist())
    if 'xgb_imp' in locals():
        print("XGBoost:", xgb_imp['feature'].head(5).tolist())
    if 'lgb_imp' in locals():
        print("LightGBM:", lgb_imp['feature'].head(5).tolist())

In [ ]:
## 📚 Bonus: Key Terms Glossary

"""
| Term | Definition | Analogy |
|------|------------|---------|
| **PM2.5** | Fine particles <2.5 micrometers that can enter lungs | Microscopic dust you can't see |
| **AQI** | Air Quality Index (1-5 scale) | Temperature: "Hot, warm, cool, cold" |
| **Feature** | Input variable used for prediction | Ingredients in a recipe |
| **Target** | What we're trying to predict | The final dish we want to cook |
| **Correlation** | How two variables move together | As ice cream sales ↑, sunglasses sales ↑ |
| **R² Score** | How much variance we explain (0-1) | How well your recipe recreates the original |
| **MAE** | Average prediction error | Average cooking time error in minutes |
| **RMSE** | Penalizes large errors | We care more if dinner is 1hr vs 5min late |
| **Overfitting** | Model memorizes instead of learns | Memorizing answers instead of understanding concepts |
| **Underfitting** | Model is too simple | A recipe that works for all dishes (poorly) |

**🎯 Key Insight:** The best model isn't always the most complex - it's the one that generalizes best to new data!
"""

"""
📋 How to Use This Enhanced Notebook
Read the markdown cells (the text explanations) to understand the "why"

Run each code cell to see the results

Observe the visualizations and read the "Interpretation" sections

Check the key terms glossary when you encounter unfamiliar concepts

Experiment! Try changing parameters or adding new features

🎓 Learning Outcomes
By the end of this notebook, you'll understand:

✅ How to explore and visualize environmental data
✅ How weather impacts air pollution
✅ What correlations tell us about relationships
✅ How to prepare data for machine learning
✅ The differences between linear, tree-based, and boosting models
✅ How to compare and select the best model
✅ How to interpret model results

Would you like me to add any additional educational content or explanations to specific cells?

"""
